In [1]:
from sentence_transformers import SentenceTransformer

# Load the Sentence-Transformer model
MODEL = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL)

In [2]:
import pandas as pd

# Load the dataset
dataset = pd.read_csv("data\Mastodon_instances_metadata_v3.csv")

# Keep rows that have at least one description field
df = dataset.dropna(subset=["description", "extended_description"], how='all')

<>:4: SyntaxWarning: invalid escape sequence '\M'
<>:4: SyntaxWarning: invalid escape sequence '\M'
C:\Users\alelc\AppData\Local\Temp\ipykernel_30716\2299482114.py:4: SyntaxWarning: invalid escape sequence '\M'
  dataset = pd.read_csv("data\Mastodon_instances_metadata_v3.csv")


In [3]:
df.describe()

,active_users
count,2630.000000
mean,270.374905
std,5377.047259
min,1.000000
25%,2.000000
50%,8.000000
75%,45.750000
max,269627.000000


In [4]:
import regex as re
from bs4 import BeautifulSoup

def clean_html(text):
    # Step 1: Parse HTML content
    soup = BeautifulSoup(text, "html.parser")

    # Step 2: Extract plain text from the parsed HTML (this also removes tags)
    clean_text = soup.get_text()

    # Step 3: Remove emojis using a regular expression
    # This regex allows all alphabets (any script), numbers, and spaces but removes emojis and symbols.
    clean_text = re.sub(r'[^\p{L}\p{N}\s]+', '', clean_text)  # \p{L} matches any letter, \p{N} matches any number

    # Step 4: Remove extra spaces, newlines, and tabs
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

In [5]:
descr_dict = {}

for i, row in df.iterrows():
    description = clean_html(row['description']) if pd.notna(row['description']) else ""
    extended_description = clean_html(row['extended_description']) if pd.notna(row['extended_description']) else ""
    combined_text = (description + " " + extended_description).strip()
    descr_dict[row['instance']] = combined_text

In [6]:
import os
import pickle

embedding_cache_path = "embeddings/sentence_transformers_emb.pkl"

instances = list(descr_dict.keys())
descriptions = list(descr_dict.values())

if not os.path.exists(embedding_cache_path):
    embeddings = model.encode(descriptions, show_progress_bar=True, convert_to_numpy=True)

    embedding_dict = {
        instances[i]: embeddings[i] 
        for i in range(len(instances))
    }

    with open(embedding_cache_path, "wb") as fOut:
        pickle.dump(embedding_dict, fOut)

else:
    with open(embedding_cache_path, "rb") as fIn:
        embedding_dict = pickle.load(fIn)

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

In [7]:
instance = "planetearth.social"
embedding = embedding_dict[instance]

print(f"Instance: {instance}, Embeddings: {embedding}")

Instance: planetearth.social, Embeddings: [-1.10214828e-02 -5.07710576e-02  8.61756951e-02  4.01439331e-02
  1.25878127e-02 -5.41435927e-02 -7.31717981e-03  3.07416208e-02
 -3.92985344e-02  2.43039634e-02 -3.97023521e-02 -1.30216524e-01
 -6.27993271e-02  3.30374576e-02  1.32262297e-02  6.78154901e-02
 -3.97209227e-02 -1.17044775e-02 -3.54783460e-02  1.41533464e-02
 -3.16840447e-02  7.77580542e-04 -1.45763876e-02  2.31694020e-02
 -1.12639561e-01  2.55688354e-02 -6.96646124e-02  1.88853107e-02
 -4.03161347e-03  2.90995389e-02  2.33360529e-02  3.70080844e-02
  5.14845960e-02 -1.66536570e-02 -5.63362520e-03  1.06229894e-01
  1.37029039e-02 -4.36208136e-02 -7.47877639e-03 -2.85244770e-02
  3.65769193e-02 -7.81144425e-02  1.72088146e-02 -5.15722260e-02
 -7.74035379e-02 -3.15632708e-02 -3.12869623e-02 -1.97788402e-02
  2.76176166e-02 -5.23500144e-02  1.15283085e-02 -1.34079561e-01
  4.10050526e-02 -1.77375171e-02 -6.22818545e-02  2.66452674e-02
 -3.88190858e-02 -4.56419811e-02  4.55519594e-02

In [8]:
import numpy as np


def find_most_similar(query_embedding, embeddings, instances, top_k=10):
    """Find the most similar embeddings to a query embedding and return their names"""
    # Calculate cosine similarity (dot product for normalized embeddings)
    similarities = np.dot(embeddings, query_embedding)
    top_indices = np.argsort(similarities)[-top_k-1:][::-1]
    
    # Return the instance names and their similarity scores
    top_instances = [instances[idx] for idx in top_indices][1::]
    top_similarities = similarities[top_indices][1::]
    
    return top_instances, top_similarities

res = find_most_similar(embedding, embeddings, instances)

print(res)

(['homestead.social', 'solarsystem.social', 'universeodon.com', 'ecoevo.social', 'mastoart.social', 'astrodon.social', 'mastodon.monoceros.co.za', 'stranger.social', 'cosmos.social', 'mastodon.cosmicnation.co'], array([0.6974356 , 0.68761677, 0.6760629 , 0.67493474, 0.6698375 ,
       0.6395473 , 0.6388748 , 0.6384341 , 0.6366468 , 0.6292464 ],
      dtype=float32))
